# Customer Churn Prediction using a Simple Neural Network (ANN)

**Goal:** Predict which telecom customers are likely to leave (`Churn = Yes`) so the business can take action to keep them.

**Dataset:** Telco Customer Churn (Kaggle) - 7,043 customers, 21 columns. Target column: `Churn`.
Download: https://www.kaggle.com/datasets/blastchar/telco-customer-churn
(file name: `WA_Fn-UseC_-Telco-Customer-Churn.csv`)

**Libraries:** Pandas, NumPy, Matplotlib/Seaborn (exploration), Scikit-Learn (preprocessing, neural network, evaluation).

**Steps**
1. Load the data
2. Data exploration (with charts)
3. Data preprocessing
4. Build the neural network (`MLPClassifier`)
5. Evaluate the model
6. Conclusions

## 1. Import libraries and load the data

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report,
                             ConfusionMatrixDisplay, RocCurveDisplay)

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
# Put the downloaded CSV in the same folder as this notebook (or change the path).
# In Google Colab, upload the file first (left sidebar > Files) and keep this path.
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
print("Rows, columns:", df.shape)
df.head()

## 2. Data exploration

### 2.1 Structure of the data

In [ ]:
df.info()

In [ ]:
# Summary statistics for the numeric columns
df.describe()

In [ ]:
# Missing values, duplicates, and number of unique values per column
print("Duplicate rows:", df.duplicated().sum())
print()
summary = pd.DataFrame({
    "missing": df.isnull().sum(),
    "unique_values": df.nunique(),
    "dtype": df.dtypes.astype(str)
})
summary

**Note:** `TotalCharges` is stored as text (object) even though it should be a number. Some rows contain a blank space instead of a value, so it needs to be cleaned during preprocessing.

In [ ]:
# Check the blank TotalCharges values
blank_total = df[df["TotalCharges"].astype(str).str.strip() == ""]
print("Rows with blank TotalCharges:", len(blank_total))
blank_total[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

### 2.2 Target variable: how many customers churn?

In [ ]:
churn_counts = df["Churn"].value_counts()
churn_pct = df["Churn"].value_counts(normalize=True) * 100
print(pd.DataFrame({"count": churn_counts, "percent": churn_pct.round(2)}))

fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(data=df, x="Churn", palette=["#4C78A8", "#E45756"], ax=ax)
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom")
ax.set_title("Customer Churn Distribution")
plt.show()

The classes are **imbalanced**: far more customers stay than leave. Because of this, accuracy alone can be misleading, so we also look at precision, recall, F1-score and ROC-AUC.

### 2.3 Churn by categorical features

In [ ]:
cat_features = ["Contract", "InternetService", "PaymentMethod", "TechSupport",
                "OnlineSecurity", "PaperlessBilling", "SeniorCitizen", "Partner"]

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
for ax, col in zip(axes.ravel(), cat_features):
    sns.countplot(data=df, x=col, hue="Churn", palette=["#4C78A8", "#E45756"], ax=ax)
    ax.set_title(f"Churn by {col}")
    ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

In [ ]:
# Churn RATE (%) for each category, easier to compare than raw counts
plot_df = df.copy()
plot_df["ChurnFlag"] = (plot_df["Churn"] == "Yes").astype(int)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, col in zip(axes, ["Contract", "InternetService", "PaymentMethod"]):
    rate = plot_df.groupby(col)["ChurnFlag"].mean().sort_values(ascending=False) * 100
    sns.barplot(x=rate.index, y=rate.values, palette="Reds_r", ax=ax)
    ax.set_title(f"Churn rate (%) by {col}")
    ax.set_ylabel("Churn rate (%)")
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

### 2.4 Churn by numeric features

In [ ]:
# TotalCharges needs to be converted to numbers before plotting
plot_df["TotalCharges"] = pd.to_numeric(plot_df["TotalCharges"], errors="coerce")

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
for i, col in enumerate(["tenure", "MonthlyCharges", "TotalCharges"]):
    sns.histplot(data=plot_df, x=col, hue="Churn", kde=True, palette=["#4C78A8", "#E45756"],
                 ax=axes[0, i])
    axes[0, i].set_title(f"Distribution of {col} by Churn")
    sns.boxplot(data=plot_df, x="Churn", y=col, palette=["#4C78A8", "#E45756"], ax=axes[1, i])
    axes[1, i].set_title(f"{col} by Churn")
plt.tight_layout()
plt.show()

### 2.5 Correlation between numeric features

In [ ]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen", "ChurnFlag"]
plt.figure(figsize=(6, 5))
sns.heatmap(plot_df[num_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1)
plt.title("Correlation Heatmap (numeric features)")
plt.show()

**Key observations from exploration**
- About 26-27% of customers churn (imbalanced target).
- Customers on **month-to-month contracts** churn far more than those on one- or two-year contracts.
- Customers with **short tenure** and **higher monthly charges** are more likely to leave.
- **Fiber optic** internet users and customers paying by **electronic check** churn more often.
- Customers **without tech support or online security** churn more often.

## 3. Data preprocessing

### 3.1 Clean the data

In [ ]:
data = df.copy()

# 1) Drop the ID column (it has no predictive value)
data = data.drop(columns=["customerID"])

# 2) Convert TotalCharges to numeric; blanks become NaN
data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")
print("Missing TotalCharges before fix:", data["TotalCharges"].isnull().sum())

# 3) Handle missing values. These customers have tenure = 0 (brand new), so they
#    have not been billed yet. A total charge of 0 is the logical value.
data["TotalCharges"] = data["TotalCharges"].fillna(0)
print("Missing TotalCharges after fix:", data["TotalCharges"].isnull().sum())

# 4) Remove duplicate rows if any
data = data.drop_duplicates()

# 5) Encode the target: Yes -> 1, No -> 0
data["Churn"] = data["Churn"].map({"Yes": 1, "No": 0})

print("Remaining missing values in the whole dataset:", data.isnull().sum().sum())
print("Shape after cleaning:", data.shape)

### 3.2 Encode categorical features

In [ ]:
# Several columns contain "No internet service" / "No phone service".
# They mean the same thing as "No", so we merge them to keep the features simple.
for col in ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport",
            "StreamingTV", "StreamingMovies", "MultipleLines"]:
    data[col] = data[col].replace({"No internet service": "No", "No phone service": "No"})

# One-hot encode all remaining text columns (drop_first avoids redundant columns)
categorical_cols = data.select_dtypes(include=["object", "string"]).columns.tolist()
print("Categorical columns being encoded:", categorical_cols)

data_encoded = pd.get_dummies(data, columns=categorical_cols, drop_first=True, dtype=int)
print("Shape after encoding:", data_encoded.shape)
data_encoded.head()

### 3.3 Split into train and test sets, then scale

In [ ]:
X = data_encoded.drop(columns=["Churn"])
y = data_encoded["Churn"]

# Stratify keeps the same churn percentage in both the train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Training set:", X_train.shape, "| Test set:", X_test.shape)
print("Churn rate - train: {:.2%}, test: {:.2%}".format(y_train.mean(), y_test.mean()))

In [ ]:
# Neural networks work best when the inputs are on a similar scale.
# Fit the scaler on the TRAINING data only, then apply it to both sets
# (this avoids leaking information from the test set).
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Mean of scaled training features (should be ~0):", X_train_scaled.mean().round(4))
print("Std of scaled training features (should be ~1):", X_train_scaled.std().round(4))

## 4. Model building: simple neural network

We use scikit-learn's `MLPClassifier` (Multi-Layer Perceptron), a feed-forward neural network:

- **Input layer:** one neuron for each feature
- **Hidden layer 1:** 32 neurons, ReLU activation
- **Hidden layer 2:** 16 neurons, ReLU activation
- **Output layer:** 1 neuron with a sigmoid function that gives the probability of churn
- **Optimizer:** Adam, with `early_stopping=True` to stop training when the validation score stops improving (reduces overfitting)

In [ ]:
mlp = MLPClassifier(
    hidden_layer_sizes=(32, 16),
    activation="relu",
    solver="adam",
    alpha=0.001,               # L2 regularization
    learning_rate_init=0.001,
    batch_size=64,
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=15,
    random_state=RANDOM_STATE,
)

mlp.fit(X_train_scaled, y_train)
print("Training stopped after", mlp.n_iter_, "iterations")
print("Number of layers:", mlp.n_layers_, "| Output activation:", mlp.out_activation_)

In [ ]:
# Learning curves: training loss and validation accuracy over the epochs
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(mlp.loss_curve_)
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Loss")

axes[1].plot(mlp.validation_scores_, color="green")
axes[1].set_title("Validation Accuracy")
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("Accuracy")
plt.tight_layout()
plt.show()

## 5. Model evaluation

In [ ]:
y_pred = mlp.predict(X_test_scaled)
y_prob = mlp.predict_proba(X_test_scaled)[:, 1]

train_acc = accuracy_score(y_train, mlp.predict(X_train_scaled))
test_acc = accuracy_score(y_test, y_pred)

print(f"Training accuracy : {train_acc:.4f}")
print(f"Test accuracy     : {test_acc:.4f}")
print(f"Precision (churn) : {precision_score(y_test, y_pred):.4f}")
print(f"Recall (churn)    : {recall_score(y_test, y_pred):.4f}")
print(f"F1-score (churn)  : {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC           : {roc_auc_score(y_test, y_prob):.4f}")
print()
print(classification_report(y_test, y_pred, target_names=["Stayed (0)", "Churned (1)"]))

In [ ]:
# Confusion matrix and ROC curve
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["Stayed", "Churned"]).plot(
    ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("Confusion Matrix (threshold = 0.5)")

RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[1], name="Neural Network")
axes[1].plot([0, 1], [0, 1], "k--", label="Random guess")
axes[1].set_title("ROC Curve")
axes[1].legend()
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True negatives: {tn} | False positives: {fp} | False negatives: {fn} | True positives: {tp}")

### 5.1 Adjusting the decision threshold

The default threshold of 0.5 often misses many churners when the classes are imbalanced. For a business, **missing a customer who will leave (false negative) is usually more costly** than contacting a customer who would have stayed (false positive). Below we test lower thresholds to see the trade-off between precision and recall.

In [ ]:
rows = []
for t in [0.5, 0.45, 0.4, 0.35, 0.3]:
    pred_t = (y_prob >= t).astype(int)
    rows.append({
        "threshold": t,
        "accuracy": accuracy_score(y_test, pred_t),
        "precision": precision_score(y_test, pred_t),
        "recall": recall_score(y_test, pred_t),
        "f1": f1_score(y_test, pred_t),
    })
threshold_table = pd.DataFrame(rows).round(3)
threshold_table

In [ ]:
# Pick the threshold with the best F1-score and show the results
best_t = threshold_table.sort_values("f1", ascending=False).iloc[0]["threshold"]
y_pred_best = (y_prob >= best_t).astype(int)
print("Best threshold by F1-score:", best_t)
print(classification_report(y_test, y_pred_best, target_names=["Stayed (0)", "Churned (1)"]))

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_best),
                       display_labels=["Stayed", "Churned"]).plot(cmap="Blues", colorbar=False)
plt.title(f"Confusion Matrix (threshold = {best_t})")
plt.show()

### 5.2 Comparison with a simple baseline (Logistic Regression)

In [ ]:
logreg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
logreg.fit(X_train_scaled, y_train)
lr_pred = logreg.predict(X_test_scaled)
lr_prob = logreg.predict_proba(X_test_scaled)[:, 1]

comparison = pd.DataFrame({
    "Model": ["Neural Network (threshold 0.5)", f"Neural Network (threshold {best_t})", "Logistic Regression"],
    "Accuracy": [accuracy_score(y_test, y_pred), accuracy_score(y_test, y_pred_best), accuracy_score(y_test, lr_pred)],
    "Precision": [precision_score(y_test, y_pred), precision_score(y_test, y_pred_best), precision_score(y_test, lr_pred)],
    "Recall": [recall_score(y_test, y_pred), recall_score(y_test, y_pred_best), recall_score(y_test, lr_pred)],
    "F1": [f1_score(y_test, y_pred), f1_score(y_test, y_pred_best), f1_score(y_test, lr_pred)],
    "ROC-AUC": [roc_auc_score(y_test, y_prob), roc_auc_score(y_test, y_prob), roc_auc_score(y_test, lr_prob)],
}).round(3)
comparison

## 6. Conclusions

- **Data:** The dataset has 7,043 customers and about 26-27% churn. The only data problem was `TotalCharges` stored as text with 11 blank values for brand-new customers (tenure = 0), which we filled with 0.
- **Exploration:** Churn is highest for month-to-month contracts, short tenure, higher monthly charges, fiber optic internet and electronic check payments.
- **Model:** A simple neural network (two hidden layers of 32 and 16 neurons) trained on scaled and one-hot encoded features, using early stopping to avoid overfitting.
- **Performance:** Look at the test accuracy, ROC-AUC, precision and recall printed above. Accuracy is usually around 80% for this dataset, but because only about a quarter of customers churn, **recall and F1 for the churn class matter more than accuracy**.
- **Threshold tuning:** Lowering the threshold below 0.5 catches more churners (higher recall) at the cost of more false alarms (lower precision). The right choice depends on the cost of a retention offer compared with the cost of losing a customer.
- **Baseline comparison:** The neural network performs about the same as logistic regression on this small tabular dataset, which is common. Neural networks usually show their advantage on larger or more complex data.
- **Next steps:** Try different hidden layer sizes, use `GridSearchCV` to tune hyperparameters, handle class imbalance with SMOTE or `class_weight`, and test other models such as Random Forest or Gradient Boosting.